In [0]:
%sql
select * from students_cleaned;

In [0]:
%sql
DESCRIBE students_cleaned;

In [0]:
%python
df = spark.read.table("students_cleaned")

In [0]:
%python
from pyspark.sql.functions import row_number, col
from pyspark.sql.window import Window

window_spec = Window.partitionBy("department").orderBy(col("cgpa").desc())

df_ranked = df.withColumn("rank", row_number().over(window_spec))

top_students = df_ranked.filter(col("rank") == 1)

top_students.display()




In [0]:
%python
from pyspark.sql.functions import lag

window_spec = Window.partitionBy("department").orderBy(col("cgpa").desc())

df_lag = df.withColumn("prev_cgpa", lag("cgpa").over(window_spec)) \
           .withColumn("cgpa_diff", col("cgpa") - col("prev_cgpa"))

df_lag.display()

In [0]:
%python
from pyspark.sql.functions import count, when, col

placement_stats = df.groupBy("department").agg(
    count("*").alias("total_students"),
    count(when(col("placement_status") == "Placed", True)).alias("placed_students")
)

placement_stats = placement_stats.withColumn(
    "placement_rate_percentange",
    (col("placed_students") / col("total_students")) * 100
)
placement_stats.display()

In [0]:
%python
window_spec = Window.orderBy(col("cgpa").desc())

rnk = df.withColumn("global_rank", row_number().over(window_spec))
rnk.display()

In [0]:
%python
from pyspark.sql.functions import col, count, when 

total_count = df.count()

null_profile = df.select([
  (count(when(col(c).isNull(), c)) / total_count).alias(c + "_null_pct") for c in df.columns
])
print(total_count)

In [0]:
%python
df.write.format("delta").mode("overwrite").saveAsTable("students_bronze")

In [0]:
%sql
SELECT * FROM students_bronze;


In [0]:
DESCRIBE DETAIL students_bronze

In [0]:
DESCRIBE FORMATTED students_bronze;

In [0]:
DESCRIBE HISTORY students_bronze;

In [0]:
CREATE TABLE students_raw (
  student_id STRING,
  full_name STRING,
  email STRING,
  phone STRING,
  city STRING,
  department STRING,
  cgpa DOUBLE,
  sports STRING,
  placement_status STRING
)
USING DELTA;

In [0]:
CREATE TABLE students_rawwww
USING DELTA
AS SELECT * FROM students_bronze;

In [0]:
SELECT * FROM students_rawwww;

In [0]:
INSERT INTO students_bronze VALUES
('101','John Doe','john@gmail.com','9999999999','Pune','CS',9.1,'Cricket','Placed');

In [0]:
INSERT INTO students_rawwww
SELECT * FROM students_bronze;

In [0]:
SELECT * FROM students_bronze VERSION AS OF 1;

In [0]:
DELETE FROM students_bronze
WHERE student_id = '101';

In [0]:
OPTIMIZE students_bronze;

In [0]:
CREATE TABLE students_clone
SHALLOW CLONE students_bronze;

In [0]:
select * from students_clone

In [0]:
CACHE TABLE students_bronze;